**Human Movement Intelligence | Gait Analysis | Presented by: Reza Saadatyar (2026)**

**04 | IMU Event Baselines and Grouped Validation**

![IMU Event Baselines and Grouped Validation](../../images/04_imu_event_baselines_and_grouped_validation.svg)

**Objective:** Compare a fixed foot/shank IMU peak detector with a Development-tuned detector using independent participant folds and matched force references.

**Dataset:** Camargo et al. (2021), source-validated original-time paired IMU and independent force-derived contact references.

**Notebook scope:** Development/Test boundaries and stated measurement units are preserved. Numerical results are only established after original-data execution.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

Read only the current source-aligned Development records from Notebook 02. Notebook 04 does not load reserved Test sensor arrays for tuning.


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("gait_analysis")
NOTEBOOK_NUMBER = 4

# ----------------- 2. Scientific Computing and Data Analysis -----------------
from pathlib import Path
import json
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------------ 4. Study Research Utilities ------------------------
from gait_experiments import (load_trial_collection, evaluate_trial_collection,
    choose_rule_config, participant_selection_score, FIXED_BASELINE)
from gait_evaluation import participant_metrics

# ---------------------- 5. Source and Output Locations -----------------------
study_dir = root / "gait_analysis"
output_dir = study_dir / "output"
cache_dir = root / "data/processed/camargo/paired_trials"
eligible = pd.read_csv(output_dir / "02_eligible_trials.csv")
roles = pd.read_csv(output_dir / "02_participant_roles.csv")
folds = pd.read_csv(output_dir / "02_development_folds.csv")
dev = eligible.merge(
    roles.loc[roles.role == "Development",["subject_id"]], on="subject_id"
)
dev = dev.merge(folds, on="subject_id", validate="many_to_one")
if dev.empty or dev.subject_id.nunique() < 5:
    raise ValueError("Insufficient Development people; complete Notebook 02.")
print("Development participants:", dev.subject_id.nunique(),
      "outer folds:", dev.validation_fold.nunique())


**02. Define Fixed Timing and Selection Metrics**

The primary endpoint is participant-mean absolute IC timing error in milliseconds for matched events, always reported with event sensitivity, precision and missed/extra counts. The detector-selection score penalises missed and extra events so predicting few contacts cannot win.


In [ ]:
# ======================== 02. Set Comparison Metrics =========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
MATCH_TOLERANCE_S = 0.15  # Predeclared candidate; fixed before Test
SEED = 42
print("One-to-one matching tolerance:", 1000*MATCH_TOLERANCE_S, "ms")
print("Rule B0:", FIXED_BASELINE)
print("B1 candidate grid is defined in gait_experiments.parameter_grid")


**03. Evaluate Development Outer Folds**

For each held-out Development fold, tune B1 using the other Development participants only. B0 uses its fixed parameters. The same Validation trials are scored by both models. No final Test labels or signals are examined.


In [ ]:
# ==================== 03. Grouped Development Evaluation =====================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
rows, configs = [], []
for fold in sorted(dev.validation_fold.unique()):
    training = dev.loc[dev.validation_fold != fold]
    validation = dev.loc[dev.validation_fold == fold]
    tr = load_trial_collection(cache_dir, training)
    va = load_trial_collection(cache_dir, validation)
    fitted = choose_rule_config(tr, MATCH_TOLERANCE_S)
    configs.append({"validation_fold": int(fold), **fitted})
    for model_id, configuration in [
        ("B0_fixed", FIXED_BASELINE), ("B1_tuned", fitted)
    ]:
        scored = evaluate_trial_collection(
            va, configuration, model_id, MATCH_TOLERANCE_S
        )
        scored["validation_fold"] = int(fold)
        rows.append(scored)
dev_scores = pd.concat(rows, ignore_index=True)
pd.DataFrame(configs).to_csv(
    output_dir / "04_fold_configurations.csv", index=False
)

# ------------------------- 2. Save Scientific Tables -------------------------
dev_scores.to_csv(output_dir / "04_development_trial_metrics.csv", index=False)
display(dev_scores.head())


**04. Participant-Level Model Comparison**

Aggregate nested trials at the independent participant level and compare both models on matched people. Report detection coverage alongside timing MAE; negative or inconclusive results are preserved.


In [ ]:
# ====================== 04. Compare Participant Metrics ======================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
participants = []
for model, frame in dev_scores.groupby("model_id", sort=True):
    participants.append(
        participant_metrics(frame).assign(model_id=model)
    )
person_results = pd.concat(participants, ignore_index=True)
person_results.to_csv(
    output_dir / "04_development_participant_metrics.csv", index=False
)
summary = (
    person_results.groupby("model_id", as_index=False)
        .agg(mean_subject_timing_mae_ms=("timing_mae_ms", "mean"),
             mean_subject_sensitivity=("sensitivity", "mean"),
             mean_subject_precision=("precision", "mean"),
             n_participants=("subject_id", "nunique"))
)
summary.to_csv(output_dir / "04_development_model_summary.csv", index=False)
display(summary)


**05. Freeze Selected Model and Configuration**

Choose the lower Development participant-balanced selection score. Independently tune the B1 final configuration on all Development people; it is never chosen using Test labels. Record the exact finalist and configuration for Notebook 05.


In [ ]:
# ======================= 05. Freeze Selected Detector ========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
model_scores = {
    model: participant_selection_score(frame)
    for model, frame in dev_scores.groupby("model_id")
}
selected = min(model_scores, key=model_scores.get)
selected_b1 = choose_rule_config(
    load_trial_collection(cache_dir, dev), MATCH_TOLERANCE_S
)
choice = FIXED_BASELINE if selected == "B0_fixed" else selected_b1
selection = {
    "selected_model_id": selected, "configuration": choice,
    "match_tolerance_s": MATCH_TOLERANCE_S,
    "development_people": int(dev.subject_id.nunique()),
    "selection_score": model_scores[selected]
}
# A compact scientific configuration consumed by Notebook 05, not a gate.
pd.DataFrame([{
    "selected_model_id": selected, "configuration_json": json.dumps(choice),
    "match_tolerance_s": MATCH_TOLERANCE_S,
    "development_people": int(dev.subject_id.nunique()),
    "selection_score": model_scores[selected]
}]).to_csv(output_dir / "04_selected_configuration.csv", index=False)
display(pd.DataFrame([selection]))


**06. Visualise Development Comparison**

Display participant-level timing errors and event coverage for the fixed and tuned detectors without implying these are final Test outcomes.


In [ ]:
# ======================= 06. Plot Development Evidence =======================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
fig, axes = plt.subplots(1, 2, figsize=(11,4.1))
for name, group in person_results.groupby("model_id"):
    axes[0].scatter([name]*len(group), group.timing_mae_ms,
                    label=name, alpha=0.6)
    axes[1].scatter([name]*len(group), group.sensitivity,
                    label=name, alpha=0.6)
axes[0].set(ylabel="Matched-event timing MAE (ms)", title="Development people")
axes[1].set(ylabel="Event sensitivity", title="Reference event coverage")
for ax in axes:
    ax.grid(alpha=0.2)

# ------------------------ 2. Export Scientific Figure ------------------------
fig.tight_layout()
save_figure(fig, output_dir / "04_grouped_development_comparison.png")
plt.show()
print("Frozen model:", selected, "configuration:", choice)


**07. Research Summary and Output Download**

The saved `04_` results compare event detectors in participant-grouped Development folds and retain the frozen model configuration. Timing MAE must be interpreted alongside sensitivity, precision and missed/extra events; reserved Test people are not used for model choice.

The shared helper packages the saved `04_` research tables and figures into one ZIP **outside the repository**. Retrieve it from Colab's file browser. Raw Camargo archives are not included, and no output is pushed to GitHub automatically.


In [ ]:
# ================= 07. Research Summary and Output Download ==================

# ------------------- 1. Review Current Scientific Outputs --------------------
print('Development comparison and frozen configuration saved to:', output_dir)

# ------------------- 2. Package Notebook-Specific Evidence -------------------
download_outputs_from_colab(
    folder="gait_analysis/output",
    stage=f"{NOTEBOOK_NUMBER:02d}",
)
